# Introduction to Physics Informed Neural Networks (PINNs)

[Full course sequence](../../ai4sci/README.md) | **2/9 · PINN fundamentals** | Previous: [PhysicsNeMo introduction](Getting_Started_PhysicsNeMo.ipynb) | Next: [Projectile](../projectile/Getting_Started_Projectile.ipynb)

This notebook targets **nvidia-physicsnemo==2.2.2**. It preserves the course problems and sequence while using the current API for models, physics residuals, losses, and training loops. A successful short run does not certify convergence or physical accuracy. Review the fixed evaluation errors in `metrics.json`, the training history in `loss.csv`, and the prediction plots together.

The `.ipynb` file provides explanations, execution cells, and result inspection; `.py` contains the actual training program; `.yaml` contains configuration. Read the examples, open the linked `.py` file in the JupyterLab editor, then **edit → save → rerun the execution cell**. Editing a Markdown code block does not change the program.

## Neural Network Solver Methodology 

In this section, we provide a brief introduction to solving differential equations with neural networks. The idea is to use a neural network to approximate the solution to the given differential equation and boundary conditions. We train this neural network by constructing a loss function for how well the neural network is satisfying the differential equation and boundary conditions. If the network can minimize this loss function, then it will in effect, solve the given differential equation.
To illustrate this idea, we will give an example of solving the following problem,
$$
\begin{align} 
    \mathbf{P} : \left\{\begin{matrix}
\frac{\mathrm{d}^2 u}{\mathrm{d} x^2}(x) = f(x), \\ 
\\
u(0) = u(1) = 0,
\end{matrix}\right.
\end{align}
$$
We start by constructing a neural network $u_{net}(x)$. The input to this network is a single value $x \in \mathbb{R}$, and its output is also a single value $u_{net}(x) \in \mathbb{R}$. We suppose that this neural network is infinitely differentiable, $u_{net} \in C^{\infty}$. The typical neural network used is a deep fully connected network where the activation functions are infinitely differentiable. 
Next, we need to construct a loss function to train this neural network. We easily encode the boundary conditions as a loss in the following way:
$$
\begin{align}
  L_{BC} = u_{net}(0)^2 + u_{net}(1)^2
\end{align}
$$
For encoding the equations, we need to compute the derivatives of $u_{net}$. Using automatic differentiation we can do so and compute $\frac{\mathrm{d}^2 u_{net}}{\mathrm{d} x^2}(x)$. This allows us to write a loss function of the form:
$$
\begin{align} 
  L_{residual} = \frac{1}{N}\sum^{N}_{i=1} \left( \frac{\mathrm{d}^2 u_{net}}{\mathrm{d} x^2}(x_i) - f(x_i) \right)^2
\end{align}
$$
Where the $x_i$'s are a batch of points sampled in the interior, $x_i \in (0, 1)$. Our total loss is $L = L_{BC} + L_{residual}$. Optimizers such as Adam are used to train this neural network. Given $f(x)=1$, the true solution is $\frac{1}{2}(x-1)x$. The original illustration below shows an example result; it is not an output or convergence guarantee from this run.
<center><img src="images/single_parabola.png" alt="Drawing" style="width:500px" /></center>

### Current PDE definition

```python
class Poisson1D(PDE):
    def __init__(self, inverse=False):
        self.dim = 1
        x = Symbol("x")
        u = Function("u")(x)
        f = Function("f")(x) if inverse else 1
        self.equations = {"poisson": u.diff(x, 2) - f}
```

### Forward PINN execution

Implementation: [pinn_basics.py](source_code/pinn_basics.py). This lab turns the original forward, parameterized, and inverse explanations into runnable exercises. The introductory examples have no blanks to complete.

`PhysicsInformer` computes derivatives with respect to $x$ and evaluates the residual. The loss contains the PDE residual and values at the two endpoints; the analytical solution is not used as a training target. Fixed evaluation coordinates are separate from training batches.

In [ ]:
import os, sys, json, subprocess, uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial").is_dir() and (p / "challenge").is_dir())
LAB = ROOT / "tutorial/introduction"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs")))
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # increase after the execution check
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
OUTPUT = OUTPUT_BASE / ("forward-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "forward"]
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

## Parameterized Problems

One important advantage of a neural network solver over traditional numerical methods is its ability to solve parameterized geometries. To illustrate this concept, we solve a parameterized version of the above problem. Suppose we want to know how the solution to this equation changes as we move the position on the boundary condition $u(l)=0$. We can parameterize this position with a variable $l \in [1,2]$ and our equation now has the form,
$$
\begin{align}
    \mathbf{P} : \left\{\begin{matrix}
\frac{\mathrm{d}^2 u}{\mathrm{d} x^2}(x) = f(x), \\ 
\\
u(0) = u(l) = 0,
\end{matrix}\right.
\end{align}
$$
To solve this parameterized problem, we can have the neural network take $l$ as input, $u_{net}(x,l)$. 
Sample $l_i\sim U(1,2)$ and $x_i\sim U(0,l_i)$. The conditional sampling density is $1/l_i$, so weight the residual by $l_i$ when approximating the domain integral.

$$L_{residual}\approx\frac1N\sum_{i=1}^N l_i\left(u_{xx}(x_i,l_i)-1\right)^2,$$
$$L_{BC}\approx\frac1N\sum_{i=1}^N\left[u(0,l_i)^2+u(l_i,l_i)^2\right].$$

The analytical solution is $u(x,l)=x(x-l)/2$. The expression above restores the missing square and closing parenthesis in the original boundary-loss formula. The problem, boundaries, and parameter range are unchanged.

![Original parameterized example](images/every_parabola.png)

In [ ]:
OUTPUT = OUTPUT_BASE / ("parameterized-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "parameterized"]
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

## Inverse Problems 

Another useful application of a neural network solver is solving inverse problems. In an inverse problem, we start with a set of observations and then use those observations to calculate the causal factors that produced them. To illustrate how to solve inverse problems with a neural network solver, we give the example of inverting out the source term $f(x)$ from the same equation from the above problem. Suppose we are given the solution $u_{true}(x)$ at 100 random points between 0 and 1, and we want to determine the $f(x)$ that is causing it. We can do this by making two neural networks $u_{net}(x)$ and $f_{net}(x)$ to approximate both $u(x)$ and $f(x)$. These networks are then optimized to minimize the following losses;
$$
\begin{align}
  L_{residual} \approx \left(\int^1_0 dx\right) \frac{1}{N} \sum^{N}_{i=1} \left(\frac{\mathrm{d}^2 u_{net}}{\mathrm{d} x^2}(x_i, l_i) - f_{net}(x_i)\right)^2
\end{align}
$$
$$
\begin{align}
  L_{data} = \frac{1}{100} \sum^{100}_{i=1} (u_{net}(x_i) - u_{true}(x_i))^2
\end{align}
$$
Using the function $u_{true}(x)=\frac{1}{48} (8 x (-1 + x^2) - (3 sin(4 \pi x))/\pi^2)$ the solution for $f(x)$ is $x + sin(4 \pi x)$. The original illustrative figures below are retained for context; compare your new results numerically.
Comparison of true solution for $f(x)$ and the function approximated by the NN:
<center><img src="images/inverse_parabola.png" alt="Drawing" style="width:500px" /><center>

Comparison of $u_{net}(x)$ and the train points from $u_{true}$:
<center><img src="images/inverse_parabola_2.png" alt="Drawing" style="width:500px" /><center>

More examples of solving an inverse problem can be found in the <a href="https://docs.nvidia.com/physicsnemo/index.html" rel="nofollow">PhysicsNeMo User Documentation</a>.
</center></center></center></center>

### Inverse PINN execution

Train the `solution` and `source` MLPs together. As in the original example, the data loss uses $u_{true}$ at 100 fixed observation points; the target values of $f$ are not used for training. Combine PDE residual, boundary, and data losses, with the data-term weight explicitly set to 100 in the code. Inspect `source_rmse` and the plot of $f$ below as well.

In [ ]:
OUTPUT = OUTPUT_BASE / ("inverse-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "inverse"]
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

In [ ]:
plt.plot(data["x"], data["source_reference"], label="true f(x)")
plt.plot(data["x"], data["source_prediction"], label="inferred f(x)")
plt.legend(); plt.show()

### Experiments and interpretation

- Increase `--steps` and check whether solution and residual errors both improve at the fixed evaluation coordinates.
- In the inverse problem, vary the number of observations and the data-loss weight; compare errors in $u$ and $f$ separately.
- A small training loss does not guarantee generalization outside the parameter range.

Use `--config` to load the same simple YAML configuration format. The CLI `--steps` argument overrides the YAML steps value.

### Next steps

[Full course sequence](../../ai4sci/README.md) | **2/9 · PINN fundamentals** | Previous: [PhysicsNeMo introduction](Getting_Started_PhysicsNeMo.ipynb) | Next: [Projectile](../projectile/Getting_Started_Projectile.ipynb)

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.